# Assignment 3 — Emotion Recognition System

Прототип face-only для Assignment 3. Запуск проекта выполняется через Docker Compose. Система оценивает выражение лица, а не устанавливает внутреннее состояние.

**Pipeline:** Input → Perception → Attention → Memory → Knowledge Representation → Output.

## 1. Подготовка Docker

В терминале из папки `ass3` выполните:

```bash
cp .env.example .env
docker compose --profile data run --rm dataset-downloader
docker compose --profile train run --rm trainer
docker compose up --build
```

После запуска сервисов можно продолжить этот notebook в профиле `notebook`. У dataset downloads могут быть ограничения Kaggle и лицензий.

In [1]:
from pathlib import Path
import json

DATA_INDEX = Path("data/processed/index.csv")
MODEL = Path("models/emotion_cnn.pt")
METRICS = Path("models/emotion_cnn.metrics.json")
ready = DATA_INDEX.exists() and MODEL.exists()
print("Data index:", DATA_INDEX.exists())
print("Trained model:", MODEL.exists())
if METRICS.exists():
    training_metrics = json.loads(METRICS.read_text())
    print("Measured test metrics:", training_metrics.get("test"))
else:
    print("Training metrics are not available yet; run the train profile.")

Data index: False
Trained model: False
Training metrics are not available yet; run the train profile.


## 2. Dataset inventory

Используются FER-2013, CK+ и RAF-DB, если загрузка и структура каталогов позволяют подготовить общие метки. Индекс содержит путь, источник, метку и split. Dataset images не сохраняются в PostgreSQL.

In [2]:
if DATA_INDEX.exists():
    import pandas as pd
    dataset_index = pd.read_csv(DATA_INDEX)
    display(dataset_index.groupby(["source", "split"]).size().unstack(fill_value=0))
    display(dataset_index.groupby(["label", "split"]).size().unstack(fill_value=0))
else:
    print("Dataset index not found. Run the data profile from the project directory.")

Dataset index not found. Run the data profile from the project directory.


## 3. Perception and Attention

Perception формирует 7 вероятностей, top label, brightness, sharpness и quality. Attention принимает кадр, только если quality не ниже 0.45 и confidence не ниже 0.55. Relevance = 0.45 × quality + 0.55 × confidence.

In [3]:
if ready:
    import cv2
    import matplotlib.pyplot as plt
    import torch
    from src.emotion_system.config import DATABASE_URL, EMOTIONS, QUALITY_THRESHOLD, CONFIDENCE_THRESHOLD, STM_WINDOW
    from src.emotion_system.model import load_model
    from src.emotion_system.pipeline import CognitiveEmotionSystem

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model, emotions = load_model(MODEL, device)
    system = CognitiveEmotionSystem(model, emotions, DATABASE_URL, QUALITY_THRESHOLD, CONFIDENCE_THRESHOLD, STM_WINDOW, device)
    sample = dataset_index[dataset_index.split == "test"].iloc[0]
    image = cv2.imread(sample.path)
    if image is None:
        raise ValueError(f"Image could not be read: {sample.path}")
    result_clear = system.process(image, "notebook-assignment3")
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()
    print("Dataset label:", sample.label)
    print("Perception:", result_clear["perception"])
    print("Attention:", result_clear["attention"])
    print("Knowledge:", result_clear["knowledge"])
else:
    print("Run dataset preparation and model training first; see Section 1.")

Run dataset preparation and model training first; see Section 1.


## 4. Память и повторный запрос

Повторим слегка ухудшенное изображение с тем же session ID. Краткосрочная память ограничена пятью результатами. PostgreSQL сохраняет metadata, а правило R2 добавляет continuity bonus только при совпадении с предыдущим принятым классом.

In [4]:
if ready:
    mildly_degraded = cv2.GaussianBlur((image.astype("float32") * 0.90).clip(0, 255).astype("uint8"), (3, 3), 0)
    result_mild = system.process(mildly_degraded, "notebook-assignment3")
    print("Memory:", result_mild["memory"])
    print("Attention:", result_mild["attention"])
    print("Knowledge:", result_mild["knowledge"])
else:
    print("Demo waits for dataset and model assets.")

Demo waits for dataset and model assets.


## 5. Слабый сигнал и отказ от решения

Сильное размытие и затемнение должны понизить качество кадра. Если критерии Attention не пройдены, правило R1 возвращает `insufficient_data` и просит получить новый вход.

In [5]:
if ready:
    heavily_degraded = cv2.GaussianBlur((image.astype("float32") * 0.03).clip(0, 255).astype("uint8"), (31, 31), 0)
    result_low = system.process(heavily_degraded, "notebook-assignment3")
    print("Perception:", result_low["perception"])
    print("Attention:", result_low["attention"])
    print("Knowledge:", result_low["knowledge"])
    assert result_low["attention"]["accepted"] is False, "The low-quality case should be rejected by Attention."
else:
    print("Demo waits for dataset and model assets.")

Demo waits for dataset and model assets.


## 6. Knowledge Representation

Граф содержит понятия Emotion, Condition, Action и MemoryFact. Правила: R1 отклоняет низкое качество; R2 применяет continuity bonus; R3 помечает низкую adjusted confidence как uncertain; R4 связывает принятый класс с действием.

In [6]:
if ready:
    import networkx as nx
    graph = system.knowledge.graph
    print("Entities:", list(graph.nodes(data=True)))
    print("Relations:", list(graph.edges(data=True)))
    pos = nx.spring_layout(graph, seed=42)
    nx.draw_networkx(graph, pos, node_size=1800, font_size=8, arrows=True)
    nx.draw_networkx_edge_labels(graph, pos, edge_labels=nx.get_edge_attributes(graph, "relation"), font_size=7)
    plt.axis("off"); plt.show()
else:
    print("Knowledge graph is constructed when the trained system is loaded.")

Knowledge graph is constructed when the trained system is loaded.


## 7. Проверки и ограничения

Выполните notebook сверху вниз после data preparation, training и запуска PostgreSQL. Сверьте фактические метрики с `models/emotion_cnn.metrics.json`. Метка тестового изображения — только ground-truth dataset label, а не вывод о внутреннем состоянии. Датасетные ограничения, качество разметки и отсутствие репрезентативности ограничивают выводы. О сырых фото система в БД не пишет.